In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 Section 5: Living With the Necessary Evil — GQA, Quantization, and the Bandwidth Fight

*Part of the Vizuara "Good and Evil of the KV Cache" series*
*Estimated time: 20–30 minutes*

## 1. Why Does This Matter?

Welcome back. In the previous sections we established something uncomfortable: the KV cache is brilliant, but it has a dark side. It trades repeated computation for repeated *data movement* — and on modern GPUs, data movement is often the scarcer resource.

Think about what happens when you use GPT-4 to draft a long legal document. Every token the model generates requires reading the *entire* KV cache back from GPU memory. For a 32,000-token context on a large model, that cache can weigh in at **tens of gigabytes** — and the model has to haul every byte of it across the memory bus for every single token it produces. The arithmetic the chip does after receiving that data is almost an afterthought.

**The real bottleneck is the truck, not the shipyard.**

So what do engineers do? Two things, and only two things are possible:

1. **Make the cache smaller** — so the truck carries less per trip
2. **Make the cache denser** — pack more information into each byte so fewer trips are needed

In this notebook we will build both families of solution from first principles:

- **Grouped-Query Attention (GQA)** — makes the cache smaller by sharing KV heads across query groups
- **Quantization** — makes each cache byte carry more information by reducing numerical precision

By the end, you will have a working implementation of both, a roofline visualizer that shows exactly *where* each technique moves you on the compute-memory tradeoff curve, and a benchmark that measures real memory and throughput on your Colab T4.

Let us begin.

## 2. Building Intuition

### The Library Analogy

Imagine a research library. The library has **bookshelves** (VRAM) and a **reading desk** (the GPU compute core). A scholar (the attention mechanism) needs to consult many books to write a single sentence.

In vanilla Multi-Head Attention (MHA), every query head has its own private bookshelf of Key and Value books. If you have 32 heads, you have 32 separate bookshelves. Every time the scholar wants to write a word, a porter must carry all 32 shelves' worth of books to the desk. The porter (memory bandwidth) is exhausted.

**Grouped-Query Attention** reorganises the library. Instead of 32 private bookshelves, we group the 32 query heads into, say, 8 groups. Each group shares *one* KV bookshelf. Now the porter only carries 8 shelves. The scholar consults slightly fewer unique books, but works 4× faster because the porter is no longer the bottleneck.

**Multi-Query Attention (MQA)** takes this to the extreme: every query head shares a *single* KV bookshelf. One shelf, maximum porter efficiency — but the scholar is consulting the same book no matter which head is asking.

### The Precision Analogy

Now think about *how* those books are written. Normally, every number in a KV cache is written in full 16-bit floating-point — like a very precise, beautifully typeset encyclopedia entry. What if we rewrote the books in a coarser shorthand — 8-bit integers, say? Each book is now half as thick. The porter can carry twice as many books per trip. The scholar's notes are slightly less precise, but for most tasks the difference is imperceptible.

That is quantization: deliberately reducing numerical precision to shrink the data that must move across the bus.

### 🤔 Think About This

Before reading further, pause and consider: if GQA reduces the number of KV heads from H to G (where G < H), but keeps all Q heads, what happens to the attention computation? The queries still need to attend to *something* — but now multiple queries attend to the *same* key and value. Does that mean we lose expressive power? When might it matter and when might it not?

Write your intuition here (no wrong answers!): ...

We will test your intuition numerically in Section 4.

## 3. The Mathematics

### 3.1 Standard Multi-Head Attention (MHA) — The Baseline

In standard MHA with $H$ heads, head dimension $d_h$, and sequence length $T$, the KV cache stores:

$$\text{Cache}_{\text{MHA}} = 2 \times H \times T \times d_h \times \text{bytes\_per\_element}$$

The factor of 2 accounts for Keys and Values. This equation says: for *every* head, *every* token position, and *every* dimension, we store two floating-point numbers. Computationally, this means that as sequence length grows, cache memory grows **linearly in both T and H** — doubling either one doubles your memory bill.

### 3.2 Grouped-Query Attention (GQA)

GQA introduces $G$ KV head groups, where $1 \leq G \leq H$. The $H$ query heads are divided evenly across $G$ groups; each group shares a single KV head pair.

$$\text{Cache}_{\text{GQA}} = 2 \times G \times T \times d_h \times \text{bytes\_per\_element}$$

The **compression ratio** relative to MHA is simply:

$$\rho_{\text{GQA}} = \frac{G}{H}$$

This equation says: we keep only a fraction $G/H$ of the original KV cache. When $G = H$ we recover MHA exactly. When $G = 1$ we have MQA — the single-group extreme.

For each query head $q$ in group $g$, attention is computed as:

$$\text{Attn}(Q_q, K_g, V_g) = \text{softmax}\!\left(\frac{Q_q K_g^{\top}}{\sqrt{d_h}}\right) V_g$$

Computationally this means: query head $q$ attends over positions using the *shared* key matrix $K_g$ of its group, and reads from the *shared* value matrix $V_g$. The queries remain independent — only the KV side is shared.

### 3.3 Quantization and the Bandwidth Equation

Let $B$ be bytes per element (2 for float16, 1 for int8, 0.5 for int4). The **effective bandwidth demand** of reading the KV cache for one generation step is:

$$\text{BW\_demand} = 2 \times G \times T \times d_h \times B \quad \text{[bytes per token generated]}$$

And the **time** spent waiting for that data, assuming peak hardware bandwidth $\Lambda$ (bytes/sec):

$$t_{\text{memory}} = \frac{\text{BW\_demand}}{\Lambda}$$

This is the floor on your per-token latency. No matter how fast your arithmetic units are, you cannot generate a token faster than $t_{\text{memory}}$. This equation makes the design space crystalline:

- **Reduce G** (GQA/MQA): shrinks BW\_demand by factor $G/H$
- **Reduce B** (quantization): shrinks BW\_demand by factor $B/B_{\text{orig}}$
- **Combine both**: multiply the savings

The two techniques are **orthogonal and multiplicative** — that is the key insight.

## 4. Let's Build It — Component by Component

### 4.1 Setup and Roofline Visualizer

Let us start by building the roofline model we have been talking about. This will be our compass throughout the notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import math
import time
from dataclasses import dataclass
from typing import Optional, Tuple

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Hardware constants (T4 GPU approximate specs) ─────────────────────────────
T4_PEAK_TFLOPS_FP16 = 65.0          # Tera-FLOPS/sec (tensor core)
T4_BANDWIDTH_TBPS   = 0.3           # Tera-Bytes/sec (~300 GB/s)
T4_RIDGE_POINT      = (T4_PEAK_TFLOPS_FP16 * 1e12) / (T4_BANDWIDTH_TBPS * 1e12)
# Ridge point = peak_flops / peak_bandwidth  (flops/byte)

print(f"T4 Peak Compute : {T4_PEAK_TFLOPS_FP16:.0f} TFLOPS (FP16)")
print(f"T4 Peak Bandwidth: {T4_BANDWIDTH_TBPS*1e3:.0f} GB/s")
print(f"Ridge Point      : {T4_RIDGE_POINT:.1f} FLOP/byte")
print("\nIf your operation does fewer than this many FLOPs per byte moved,")
print("you are memory-bound. LLM token generation typically does ~1-10 FLOP/byte.")

In [ ]:
def plot_roofline(points: list[dict], title: str = "Roofline Model — T4 GPU") -> None:
    """
    Plot a roofline diagram and overlay labeled operating points.

    Each point dict: {'label': str, 'ai': float (FLOP/byte), 'color': str}
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # ── Draw roofline ─────────────────────────────────────────────────────────
    ai_vals = np.logspace(-2, 4, 500)
    peak_compute  = T4_PEAK_TFLOPS_FP16 * 1e12          # FLOP/s
    peak_bw       = T4_BANDWIDTH_TBPS   * 1e12           # byte/s

    # Achieved performance = min(ai * peak_bw, peak_compute)
    perf = np.minimum(ai_vals * peak_bw, peak_compute)

    ax.loglog(ai_vals, perf / 1e12, color='steelblue', linewidth=3, label='Roofline')

    # ── Shade regions ─────────────────────────────────────────────────────────
    ax.axvspan(ai_vals[0], T4_RIDGE_POINT, alpha=0.08, color='red')
    ax.axvspan(T4_RIDGE_POINT, ai_vals[-1], alpha=0.08, color='green')
    ax.axvline(T4_RIDGE_POINT, color='gray', linestyle='--', alpha=0.6,
               label=f'Ridge Point ({T4_RIDGE_POINT:.0f} FLOP/byte)')

    ax.text(T4_RIDGE_POINT * 0.03, T4_PEAK_TFLOPS_FP16 * 0.35,
            '← Memory-Bound\n(bandwidth limited)', fontsize=10,
            color='darkred', alpha=0.85)
    ax.text(T4_RIDGE_POINT * 1.3, T4_PEAK_TFLOPS_FP16 * 0.35,
            'Compute-Bound →\n(FLOP limited)', fontsize=10,
            color='darkgreen', alpha=0.85)

    # ── Overlay operating points ───────────────────────────────────────────────
    for p in points:
        ai    = p['ai']
        perf_pt = min(ai * peak_bw, peak_compute) / 1e12
        ax.scatter(ai, perf_pt, s=200, color=p['color'], zorder=5,
                   edgecolors='black', linewidth=1.5)
        ax.annotate(p['label'], (ai, perf_pt),
                    textcoords='offset points', xytext=(8, 6), fontsize=9,
                    fontweight='bold', color=p['color'])

    ax.set_xlabel('Arithmetic Intensity (FLOP / byte moved)', fontsize=12)
    ax.set_ylabel('Achieved Performance (TFLOPS)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout()
    plt.savefig('roofline.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Roofline saved as roofline.png")

# ── Typical arithmetic intensities for KV cache read during generation ─────────
# For one decode step: ~2 FLOPs per weight byte (matmul), but KV read
# is essentially 0 FLOPs of arithmetic for many bytes moved → very low AI
roofline_points = [
    {'label': 'MHA Decode\n(T=2048)',  'ai': 1.2,  'color': 'crimson'},
    {'label': 'GQA G=8\n(T=2048)',    'ai': 4.8,  'color': 'darkorange'},
    {'label': 'MQA\n(T=2048)',        'ai': 9.6,  'color': 'gold'},
    {'label': 'GQA+INT8\n(T=2048)',   'ai': 9.6,  'color': 'limegreen'},
    {'label': 'GQA+INT4\n(T=2048)',   'ai': 19.2, 'color': 'dodgerblue'},
]

plot_roofline(roofline_points)

Notice how all these points sit on the *sloped* (memory-bound) section of the roofline. GQA and quantization slide them rightward — increasing arithmetic intensity by reducing the data they must haul — but even the best combination is still firmly bandwidth-limited at long sequence lengths. This is the fundamental challenge.

### 4.2 KV Cache Memory Calculator

Before we implement anything, let us build an exact calculator so we can see the memory stakes clearly.

In [ ]:
@dataclass
class ModelConfig:
    """Configuration for a transformer model."""
    n_layers: int
    n_heads_q: int          # number of query heads
    n_heads_kv: int         # number of KV heads (= n_heads_q for MHA, 1 for MQA)
    d_model: int
    max_seq_len: int
    dtype_bytes: float = 2.0   # 2=float16, 1=int8, 0.5=int4

    @property
    def d_head(self) -> int:
        return self.d_model // self.n_heads_q

    def kv_cache_bytes(self, seq_len: int) -> int:
        """Total KV cache bytes for a given sequence length."""
        return int(
            2               # K and V
            * self.n_layers
            * self.n_heads_kv
            * seq_len
            * self.d_head
            * self.dtype_bytes
        )

    def kv_cache_gb(self, seq_len: int) -> float:
        return self.kv_cache_bytes(seq_len) / 1e9

    def bandwidth_demand_per_token(self, seq_len: int) -> float:
        """Bytes that must be read from VRAM to generate ONE token."""
        return self.kv_cache_bytes(seq_len)  # full cache read each step


# ── Define several configurations ─────────────────────────────────────────────
llama2_7b_mha = ModelConfig(
    n_layers=32, n_heads_q=32, n_heads_kv=32,
    d_model=4096, max_seq_len=4096
)
llama2_7b_gqa = ModelConfig(
    n_layers=32, n_heads_q=32, n_heads_kv=8,
    d_model=4096, max_seq_len=4096
)
llama2_7b_mqa = ModelConfig(
    n_layers=32, n_heads_q=32, n_heads_kv=1,
    d_model=4096, max_seq_len=4096
)
llama2_7b_gqa_int8 = ModelConfig(
    n_layers=32, n_heads_q=32, n_heads_kv=8,
    d_model=4096, max_seq_len=4096, dtype_bytes=1.0
)
llama2_7b_gqa_int4 = ModelConfig(
    n_layers=32, n_heads_q=32, n_heads_kv=8,
    d_model=4096, max_seq_len=4096, dtype_bytes=0.5
)

configs = {
    'MHA (fp16)':       llama2_7b_mha,
    'GQA G=8 (fp16)':   llama2_7b_gqa,
    'MQA (fp16)':       llama2_7b_mqa,
    'GQA G=8 (int8)':   llama2_7b_gqa_int8,
    'GQA G=8 (int4)':   llama2_7b_gqa_int4,
}

# ── Print comparison table ─────────────────────────────────────────────────────
seq_lens = [1024, 4096, 16384, 32768]

print(f"{'Config':<22}", end='')
for T in seq_lens:
    print(f"  T={T//1024}K GB", end='')
print(f"\n{'─'*22}", '─'*12 * len(seq_lens))

for name, cfg in configs.items():
    print(f"{name:<22}", end='')
    for T in seq_lens:
        gb = cfg.kv_cache_gb(T)
        print(f"  {gb:>8.2f}  ", end='')
    print()

print(f"\n✦ MHA→GQA(G=8) compression ratio: {llama2_7b_gqa.n_heads_kv/llama2_7b_mha.n_heads_kv:.3f}×")
print(f"✦ fp16→int8 compression ratio: 0.500×")
print(f"✦ fp16→int4 compression ratio: 0.250×")
print(f"✦ Combined GQA+int4 vs MHA fp16: {(8/32)*0.25:.4f}× ({(32/8)*4:.0f}× smaller!)")

In [ ]:
# ── Visualise cache size vs sequence length ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_map = {
    'MHA (fp16)':     'crimson',
    'GQA G=8 (fp16)': 'darkorange',
    'MQA (fp16)':     'gold',
    'GQA G=8 (int8)': 'limegreen',
    'GQA G=8 (int4)': 'dodgerblue',
}

T_range = np.arange(512, 33000, 256)

# Left: absolute cache size
ax = axes[0]
for name, cfg in configs.items():
    gb_vals = [cfg.kv_cache_gb(int(T)) for T in T_range]
    ax.plot(T_range / 1000, gb_vals, color=colors_map[name],
            linewidth=2.5, label=name)

ax.axhline(16, color='black', linestyle=':', alpha=0.5, label='16 GB (T4 VRAM)')
ax.fill_between(T_range / 1000, 16, 40, alpha=0.07, color='red')
ax.text(5, 17.5, 'T4 VRAM limit', fontsize=9, color='gray')
ax.set_xlabel('Sequence Length (K tokens)', fontsize=11)
ax.set_ylabel('KV Cache Size (GB)', fontsize=11)
ax.set_title('KV Cache Size vs Sequence Length\n(LLaMA-2 7B architecture)', fontsize=11)
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3)

# Right: compression factor relative to MHA fp16
ax = axes[1]
mha_baseline = np.array([llama2_7b_mha.kv_cache_gb(int(T)) for T in T_range])
for name, cfg in configs.items():
    if name == 'MHA (fp16)':
        continue
    gb_vals = np.array([cfg.kv_cache_gb(int(T)) for T in T_range])
    reduction_pct = (1 - gb_vals / mha_baseline) * 100
    ax.plot(T_range / 1000, reduction_pct, color=colors_map[name],
            linewidth=2.5, label=name)

ax.set_xlabel('Sequence Length (K tokens)', fontsize=11)
ax.set_ylabel('Cache Reduction vs MHA fp16 (%)', fontsize=11)
ax.set_title('Memory Savings Relative to MHA Baseline', fontsize=11)
ax.legend(fontsize=8, loc='center right')
ax.grid(alpha=0.3)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('kv_cache_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Cache size comparison saved as kv_cache_sizes.png")

The right panel tells us something important: the savings from GQA and quantization are constant fractions — they do not depend on sequence length. A 4× GQA compression is 4× at T=1K just as much as at T=32K. This makes these techniques universally effective regardless of context length.

### 4.3 Implementing Grouped-Query Attention from Scratch

Now let us implement the actual attention mechanism. We will build MHA, GQA, and MQA as a single unified function — the only thing that changes is `n_heads_kv`.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    A clean implementation of Grouped-Query Attention (GQA) that subsumes
    Multi-Head Attention (n_kv_heads == n_q_heads) and
    Multi-Query Attention  (n_kv_heads == 1) as special cases.
    """

    def __init__(self, d_model: int, n_heads_q: int, n_heads_kv: int,
                 dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads_q == 0, "d_model must be divisible by n_heads_q"
        assert n_heads_q % n_heads_kv == 0, \
            "n_heads_q must be divisible by n_heads_kv (each KV head serves n_q/n_kv query heads)"

        self.n_q   = n_heads_q
        self.n_kv  = n_heads_kv
        self.n_rep = n_heads_q // n_heads_kv   # how many Q heads share each KV head
        self.d_h   = d_model // n_heads_q
        self.scale = self.d_h ** -0.5

        # Query projection: n_q  heads × d_h dims each
        self.W_q = nn.Linear(d_model, n_heads_q  * self.d_h, bias=False)
        # Key/Value projections: n_kv heads × d_h dims each  (smaller!)
        self.W_k = nn.Linear(d_model, n_heads_kv * self.d_h, bias=False)
        self.W_v = nn.Linear(d_model, n_heads_kv * self.d_h, bias=False)
        # Output projection
        self.W_o = nn.Linear(n_heads_q * self.d_h, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

        # ── Count parameters for display ──────────────────────────────────────
        self._n_params_kv = (n_heads_kv * self.d_h * d_model) * 2  # K and V

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """
        Expand KV from (B, n_kv, T, d_h) to (B, n_q, T, d_h)
        by repeating each KV head n_rep times.

        This is the "interleaved repeat" that pairs each KV head
        with its corresponding query group.
        """
        B, n_kv, T, d_h = x.shape
        if self.n_rep == 1:
            return x  # MHA: nothing to repeat
        # Insert a repeat dimension, then flatten back
        return (
            x[:, :, None, :, :]              # (B, n_kv, 1,     T, d_h)
             .expand(B, n_kv, self.n_rep, T, d_h)  # (B, n_kv, n_rep, T, d_h)
             .reshape(B, self.n_q, T, d_h)   # (B, n_q,  T, d_h)
        )

    def forward(
        self,
        x: torch.Tensor,                         # (B, T, d_model)
        mask: Optional[torch.Tensor] = None,     # (T, T) causal mask
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:

        B, T, _ = x.shape

        # ── Project to Q, K, V ───────────────────────────────────────────────
        Q = self.W_q(x).reshape(B, T, self.n_q,  self.d_h).transpose(1, 2)
        K = self.W_k(x).reshape(B, T, self.n_kv, self.d_h).transpose(1, 2)
        V = self.W_v(x).reshape(B, T, self.n_kv, self.d_h).transpose(1, 2)
        # Shapes: Q → (B, n_q, T, d_h)  |  K, V → (B, n_kv, T, d_h)

        # ── Append to / read from KV cache (for autoregressive generation) ───
        if kv_cache is not None:
            K_past, V_past = kv_cache
            K = torch.cat([K_past, K], dim=2)  # extend along sequence dim
            V = torch.cat([V_past, V], dim=2)

        new_cache = (K, V)

        # ── Expand KV heads to match Q heads ─────────────────────────────────
        K_exp = self._repeat_kv(K)   # (B, n_q, T_full, d_h)
        V_exp = self._repeat_kv(V)

        # ── Scaled dot-product attention ──────────────────────────────────────
        scores = (Q @ K_exp.transpose(-2, -1)) * self.scale  # (B, n_q, T, T_full)
        if mask is not None:
            scores = scores + mask

        attn = self.dropout(torch.softmax(scores, dim=-1))
        out  = attn @ V_exp              # (B, n_q, T, d_h)

        # ── Merge heads and project ───────────────────────────────────────────
        out = out.transpose(1, 2).reshape(B, T, self.n_q * self.d_h)
        out = self.W_o(out)              # (B, T, d_model)

        return out, new_cache

In [ ]:
# ── Quick sanity check ─────────────────────────────────────────────────────────
d_model = 64
B, T    = 2, 16

mha_layer = GroupedQueryAttention(d_model, n_heads_q=8, n_heads_kv=8)
gqa_layer = GroupedQueryAttention(d_model, n_heads_q=8, n_heads_kv=2)
mqa_layer = GroupedQueryAttention(d_model, n_heads_q=8, n_heads_kv=1)

x_test = torch.randn(B, T, d_model)

out_mha, cache_mha = mha_layer(x_test)
out_gqa, cache_gqa = gqa_layer(x_test)
out_mqa, cache_mqa = mqa_layer(x_test)

print("Output shapes (should all be B×T×d_model):")
print(f"  MHA : {tuple(out_mha.shape)}")
print(f"  GQA : {tuple(out_gqa.shape)}")
print(f"  MQA : {tuple(out_mqa.shape)}")

print("\nKV cache shapes [K, V]:")
for name, cache in [("MHA", cache_mha), ("GQA", cache_gqa), ("MQA", cache_mqa)]:
    k_shape = tuple(cache[0].shape)
    print(f"  {name}: K={k_shape}  (n_kv_heads={k_shape[1]})")

print("\nKV parameter counts (K + V weight matrices):")
for name, layer in [("MHA", mha_layer), ("GQA", gqa_layer), ("MQA", mqa_layer)]:
    print(f"  {name}: {layer._n_params_kv:,} params")
print("\n✅ All shapes correct!")

### 4.4 Visualising the Attention Patterns

Let us make the KV sharing concrete by visualising which query heads share which KV heads.

In [ ]:
def visualize_gqa_grouping(n_heads_q: int, n_heads_kv: int, title: str) -> None:
    """Draw a diagram showing which Q heads map to which KV heads."""
    n_rep = n_heads_q // n_heads_kv
    cmap  = plt.cm.get_cmap('tab10', n_heads_kv)

    fig, ax = plt.subplots(figsize=(max(8, n_heads_q * 0.6), 3.5))

    for q_idx in range(n_heads_q):
        kv_idx = q_idx // n_rep
        color  = cmap(kv_idx)

        # Q head box
        ax.add_patch(plt.Rectangle((q_idx, 1.4), 0.85, 0.6,
                                   facecolor=color, edgecolor='white',
                                   linewidth=2, alpha=0.85))
        ax.text(q_idx + 0.425, 1.72, f'Q{q_idx}',
                ha='center', va='center', fontsize=8, fontweight='bold', color='white')

    for kv_idx in range(n_heads_kv):
        color  = cmap(kv_idx)
        x_pos  = kv_idx * n_rep + (n_rep - 1) / 2   # center under group

        # KV head box
        ax.add_patch(plt.Rectangle((x_pos - 0.5 + 0.425, 0.2), 1.0, 0.6,
                                   facecolor=color, edgecolor='white',
                                   linewidth=2, alpha=0.65))
        ax.text(x_pos + 0.425, 0.52, f'KV{kv_idx}',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')

        # Draw arrows from Q heads in this group to the shared KV head
        for rep in range(n_rep):
            q_x = kv_idx * n_rep + rep + 0.425
            ax.annotate("", xy=(x_pos + 0.425, 0.82), xytext=(q_x, 1.38),
                        arrowprops=dict(arrowstyle='->', color=color,
                                        lw=1.5, alpha=0.7))

    ax.set_xlim(-0.3, n_heads_q + 0.2)
    ax.set_ylim(0, 2.3)
    ax.set_yticks([0.52, 1.72])
    ax.set_yticklabels(['KV Heads\n(cache)', 'Query Heads'], fontsize=10)
    ax.set_xticks([])
    ax.set_title(f'{title}\n(n_q={n_heads_q}, n_kv={n_heads_kv}, '
                 f'sharing ratio={n_rep}×)', fontsize=11, fontweight='bold')
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
    plt.tight_layout()
    plt.show()

print("── MHA (no sharing) ─────────────────────────────────────────────────────")
visualize_gqa_grouping(8, 8, "Multi-Head Attention (MHA)")

print("── GQA (4 groups) ───────────────────────────────────────────────────────")
visualize_gqa_grouping(8, 2, "Grouped-Query Attention (GQA, G=2)")

print("── MQA (one shared KV head) ─────────────────────────────────────────────")
visualize_gqa_grouping(8, 1, "Multi-Query Attention (MQA)")

### 4.5 Quantization — Shrinking the Cache Byte by Byte

Now let us implement simple round-trip quantization for the KV cache. The idea is elegant: store keys and values in lower precision, dequantize them back to float16 just before the attention computation.

In [ ]:
class QuantizedKVCache:
    """
    A KV cache that stores tensors in reduced precision (int8 or int4-simulated)
    and dequantizes them on read.

    This is an *absmax* per-tensor quantization — simple but illustrative.
    Production systems (e.g. bitsandbytes, GPTQ) use per-channel or per-group
    quantization for much better accuracy.
    """

    def __init__(self, bits: int = 8):
        assert bits in (8, 4), "Only 8-bit and 4-bit supported in this demo"
        self.bits    = bits
        self.n_levels = 2 ** bits - 1     # e.g. 255 for int8
        self._K_q: Optional[torch.Tensor] = None
        self._V_q: Optional[torch.Tensor] = None
        self._K_scale: Optional[float]    = None
        self._V_scale: Optional[float]    = None

    def _quantize(self, x: torch.Tensor) -> Tuple[torch.Tensor, float]:
        """Quantize float tensor to integer representation."""
        absmax = x.abs().max().item()
        scale  = absmax / (self.n_levels / 2)       # symmetric quantization
        if scale == 0:
            scale = 1e-8
        x_q = torch.clamp(
            torch.round(x / scale),
            -(self.n_levels // 2), (self.n_levels // 2)
        ).to(torch.int8)                            # store as int8
        return x_q, scale

    def _dequantize(self, x_q: torch.Tensor, scale: float) -> torch.Tensor:
        """Restore float tensor from integer representation."""
        return x_q.float() * scale

    def update(self, K_new: torch.Tensor, V_new: torch.Tensor):
        """Append new K, V slices to the cache in quantized form."""
        K_q, K_scale = self._quantize(K_new)
        V_q, V_scale = self._quantize(V_new)

        if self._K_q is None:
            self._K_q    = K_q
            self._V_q    = V_q
            self._K_scale = K_scale
            self._V_scale = V_scale
        else:
            # For simplicity, re-quantize the concatenated tensor
            # (real systems maintain running scale factors per token)
            K_full = torch.cat([self._dequantize(self._K_q, self._K_scale),
                                 K_new], dim=2)
            V_full = torch.cat([self._dequantize(self._V_q, self._V_scale),
                                 V_new], dim=2)
            self._K_q, self._K_scale = self._quantize(K_full)
            self._V_q, self._V_scale = self._quantize(V_full)

    def read(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return dequantized K, V tensors in float32."""
        return (
            self._dequantize(self._K_q, self._K_scale),
            self._dequantize(self._V_q, self._V_scale),
        )

    def bytes_used(self) -> int:
        """Actual bytes stored (int8 tensors)."""
        if self._K_q is None:
            return 0
        return self._K_q.numel() + self._V_q.numel()   # int8: 1 byte each

    def bytes_if_fp16(self) -> int:
        """Bytes this would occupy in float16."""
        if self._K_q is None:
            return 0
        return (self._K_q.numel() + self._V_q.numel()) * 2

In [ ]:
# ── Demonstrate quantization error vs precision ────────────────────────────────
torch.manual_seed(0)

# Simulate a small KV cache slice: 1 batch, 4 heads, 64 tokens, 16 dim
K_fp16 = torch.randn(1, 4, 64, 16, dtype=torch.float32) * 0.5
V_fp16 = torch.randn(1, 4, 64, 16, dtype=torch.float32) * 0.5

results = {}
for bits in [8, 4]:
    cache = QuantizedKVCache(bits=bits)
    cache.update(K_fp16, V_fp16)
    K_rec, V_rec = cache.read()

    k_err = (K_rec - K_fp16).abs().mean().item()
    v_err = (V_rec - V_fp16).abs().mean().item()
    mem   = cache.bytes_used()
    mem_orig = cache.bytes_if_fp16()

    results[bits] = {'k_err': k_err, 'v_err': v_err,
                     'mem': mem, 'mem_orig': mem_orig}

    print(f"INT{bits}:")
    print(f"  Mean |K error|      : {k_err:.6f}")
    print(f"  Mean |V error|      : {v_err:.6f}")
    print(f"  Memory (quantized) : {mem:,} bytes")
    print(f"  Memory (fp16)      : {mem_orig:,} bytes")
    print(f"  Compression        : {mem_orig/mem:.1f}×\n")

In [ ]:
# ── Visualise quantization error distribution ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

k_original = K_fp16.flatten().numpy()

for col, bits in enumerate([8, 4]):
    cache = QuantizedKVCache(bits=bits)
    cache.update(K_fp16, V_fp16)
    K_rec, _ = cache.read()
    k_rec = K_rec.flatten().numpy()
    k_err = (k_rec - k_original)

    # Original vs Reconstructed
    ax = axes[0, col]
    ax.scatter(k_original[:200], k_rec[:200], alpha=0.4, s=15,
               color='steelblue' if bits == 8 else 'darkorange')
    lim = max(abs(k_original).max(), abs(k_rec).max()) * 1.05
    ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1.5, label='Perfect reconstruction')
    ax.set_xlabel('Original value', fontsize=10)
    ax.set_ylabel('Reconstructed value', fontsize=10)
    ax.set_title(f'INT{bits}: Original vs Reconstructed K values', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Error histogram
    ax = axes[1, col]
    ax.hist(k_err, bins=50,
            color='steelblue' if bits == 8 else 'darkorange',
            edgecolor='white', alpha=0.85)
    ax.axvline(0, color='red', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Quantization error', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'INT{bits}: Error Distribution (σ = {k_err.std():.5f})', fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('Quantization Error: INT8 vs INT4', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('quantization_error.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Quantization error plot saved as quantization_error.png")

INT8 quantization produces tiny errors — the scatter plot is nearly diagonal. INT4 is coarser, but for typical activation distributions the errors remain small relative to the signal. This is the key empirical finding that makes KV quantization practical.

## 5. 🔧 Your Turn

### TODO: Implement the Bandwidth-Aware Token Latency Estimator

Now it is your turn to put the mathematics from Section 3 into code. We want a function that estimates how long (in milliseconds) it takes to generate one token, given that we are bandwidth-bound.

Remember the formula:

$$t_{\text{token}} = \frac{\text{BW\_demand}}{\Lambda}$$

where $\Lambda$ is the peak hardware bandwidth in bytes/second.

In [ ]:
def estimate_token_latency_ms(
    cfg: ModelConfig,
    seq_len: int,
    hardware_bandwidth_gbps: float = 300.0,
) -> float:
    """
    Estimate the minimum per-token generation latency (ms) imposed by
    memory bandwidth when reading the KV cache.

    Parameters
    ----------
    cfg : ModelConfig
        Model configuration (includes n_layers, n_heads_kv, d_head, dtype_bytes).
    seq_len : int
        Current sequence length (number of tokens in KV cache).
    hardware_bandwidth_gbps : float
        Peak memory bandwidth of the hardware in GB/s (default: T4 ≈ 300 GB/s).

    Returns
    -------
    float
        Estimated minimum latency in milliseconds to generate one token.

    Hints
    -----
    Step 1: Compute BW_demand = bytes that must be read from the KV cache
            for one generation step. (Hint: use cfg.bandwidth_demand_per_token)
    Step 2: Convert hardware bandwidth from GB/s to bytes/s.
    Step 3: t_memory = BW_demand / bandwidth_bytes_per_sec
    Step 4: Convert from seconds to milliseconds (multiply by 1000).
    """
    # ============ TODO ============
    # Step 1: Get bandwidth demand in bytes
    bw_demand_bytes = ???  # YOUR CODE HERE

    # Step 2: Convert GB/s → bytes/s
    bw_bytes_per_sec = ???  # YOUR CODE HERE

    # Step 3: Compute latency in seconds
    t_seconds = ???  # YOUR CODE HERE

    # Step 4: Convert to milliseconds
    t_ms = ???  # YOUR CODE HERE
    # ============ END TODO ========

    return t_ms


def estimate_max_throughput_tps(
    cfg: ModelConfig,
    seq_len: int,
    hardware_bandwidth_gbps: float = 300.0,
) -> float:
    """
    Estimate peak tokens-per-second throughput (bandwidth-limited).
    This is simply 1000 / t_ms (tokens per millisecond → per second).
    """
    # ============ TODO ============
    t_ms = ???  # YOUR CODE HERE — call estimate_token_latency_ms
    tps  = ???  # YOUR CODE HERE — convert ms/token → tokens/sec
    # ============ END TODO ========
    return tps

In [ ]:
# ── ✅ Verification cell ───────────────────────────────────────────────────────
# Run this once you have implemented the functions above.
# All assertions should pass.

try:
    # Test 1: MHA at T=1024 on T4
    lat_mha = estimate_token_latency_ms(llama2_7b_mha, seq_len=1024, hardware_bandwidth_gbps=300.0)
    assert isinstance(lat_mha, float), "Return type must be float"
    assert lat_mha > 0, "Latency must be positive"
    # MHA at T=1024: ~32 heads × 128 d_h × 1024 tokens × 32 layers × 2 (K+V) × 2 bytes
    # ≈ 2^32 * 32 * 128 * 1024 * 2 * 2 = 536 MB → 536e6 / 300e9 ≈ 1.79 ms
    assert 0.5 < lat_mha < 10.0, f"MHA latency {lat_mha:.3f} ms seems wrong"

    # Test 2: GQA should be ~4× faster than MHA (32 heads → 8 heads)
    lat_gqa = estimate_token_latency_ms(llama2_7b_gqa, seq_len=1024, hardware_bandwidth_gbps=300.0)
    ratio = lat_mha / lat_gqa
    assert abs(ratio - 4.0) < 0.1, f"MHA/GQA ratio should be 4.0, got {ratio:.3f}"

    # Test 3: Latency should scale linearly with seq_len
    lat_t1k = estimate_token_latency_ms(llama2_7b_gqa, seq_len=1000)
    lat_t2k = estimate_token_latency_ms(llama2_7b_gqa, seq_len=2000)
    assert abs(lat_t2k / lat_t1k - 2.0) < 0.01, "Latency must scale linearly with T"

    # Test 4: Throughput is inverse of latency
    tps = estimate_max_throughput_tps(llama2_7b_gqa, seq_len=1024)
    assert abs(tps - 1000.0 / lat_gqa) < 0.1, "TPS must be 1000/lat_ms"

    print("✅ All 4 assertions passed!")
    print(f"\n  MHA  latency @ T=1024: {lat_mha:.3f} ms")
    print(f"  GQA  latency @ T=1024: {lat_gqa:.3f} ms")
    print(f"  Speedup factor       : {ratio:.2f}×")
    print(f"  GQA throughput       : {tps:.1f} tokens/sec (bandwidth-limited)")

except NameError:
    print("⚠️  Please implement the TODO functions above first, then re-run this cell.")

Great work! If your assertions pass, you have just built a quantitative model of why engineers care so deeply about GQA and quantization — the latency differences are not aesthetic, they are measured in real milliseconds per token.

## 6. Putting It All Together

### The Complete Efficiency Dashboard

Let us now combine everything into a single comprehensive visualization — a bandwidth efficiency dashboard that shows latency, throughput, and memory for all our configurations across sequence lengths.

In [ ]:
def build_efficiency_report(
    configs: dict,
    seq_lens: list[int],
    hardware_bw_gbps: float = 300.0,
    colors: dict = None,
) -> dict:
    """Compute latency, throughput, and cache size for all configs × seq lens."""
    if colors is None:
        colors = {name: f'C{i}' for i, name in enumerate(configs)}

    report = {name: {'latency': [], 'throughput': [], 'cache_gb': []}
              for name in configs}

    for name, cfg in configs.items():
        for T in seq_lens:
            lat = estimate_token_latency_ms(cfg, T, hardware_bw_gbps)
            tps = estimate_max_throughput_tps(cfg, T, hardware_bw_gbps)
            gb  = cfg.kv_cache_gb(T)
            report[name]['latency'].append(lat)
            report[name]['throughput'].append(tps)
            report[name]['cache_gb'].append(gb)

    return report


colors_map = {
    'MHA (fp16)':     'crimson',
    'GQA G=8 (fp16)': 'darkorange',
    'MQA (fp16)':     'gold',
    'GQA G=8 (int8)': 'limegreen',
    'GQA G=8 (int4)': 'dodgerblue',
}

T_range_k = [512, 1024, 2048, 4096, 8192, 16384]
report = build_efficiency_report(configs, T_range_k, colors=colors_map)

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.35)

ax_lat  = fig.add_subplot(gs[0, 0])  # Latency vs T
ax_tps  = fig.add_subplot(gs[0, 1])  # Throughput vs T
ax_mem  = fig.add_subplot(gs[0, 2])  # Cache GB vs T
ax_bar  = fig.add_subplot(gs[1, :])  # Grouped bar at fixed T

T_labels = [f"{T//1024}K" for T in T_range_k]

for name, data in report.items():
    c = colors_map[name]
    ax_lat.plot(T_labels, data['latency'],    color=c, marker='o', linewidth=2, label=name)
    ax_tps.plot(T_labels, data['throughput'], color=c, marker='s', linewidth=2, label=name)
    ax_mem.plot(T_labels, data['cache_gb'],   color=c, marker='^', linewidth=2, label=name)

for ax, ylabel, title in [
    (ax_lat, 'Latency (ms/token)',   'Per-Token Latency\n(bandwidth-limited estimate)'),
    (ax_tps, 'Tokens / second',      'Peak Throughput\n(bandwidth-limited estimate)'),
    (ax_mem, 'KV Cache Size (GB)',   'KV Cache Memory'),
]:
    ax.set_xlabel('Sequence Length', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7, loc='best')
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

# ── Grouped bar chart: speedup at T=4096 relative to MHA ─────────────────────
T_fixed = 4096
mha_lat = estimate_token_latency_ms(llama2_7b_mha, T_fixed)
bar_names = [n for n in configs if n != 'MHA (fp16)']
speedups  = [mha_lat / estimate_token_latency_ms(cfg, T_fixed)
             for name, cfg in configs.items() if name != 'MHA (fp16)']

bars = ax_bar.bar(bar_names, speedups,
                  color=[colors_map[n] for n in bar_names],
                  edgecolor='black', linewidth=1.2, alpha=0.88)

ax_bar.axhline(1.0, color='crimson', linewidth=2, linestyle='--',
               label='MHA baseline (1×)')
for bar, sp in zip(bars, speedups):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.08,
                f'{sp:.1f}×', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax_bar.set_ylabel('Latency Speedup vs MHA fp16', fontsize=11)
ax_bar.set_title(f'Bandwidth-Limited Speedup at T={T_fixed//1024}K tokens\n'
                 '(higher = faster generation)', fontsize=11, fontweight='bold')
ax_bar.legend(fontsize=10)
ax_bar.grid(axis='y', alpha=0.3)
ax_bar.set_ylim(0, max(speedups) * 1.3)

plt.suptitle('LLaMA-2 7B Architecture — Bandwidth Efficiency Dashboard\n'
             '(T4 GPU, 300 GB/s bandwidth)', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('efficiency_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Full efficiency dashboard saved as efficiency_dashboard.png")

## 7. Benchmarking on Real Hardware

Theory is one thing. Let us actually measure the memory footprint and compute time of our GQA implementation on the T4 we are running on right now.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

def measure_attention_memory_and_time(
    n_heads_q: int,
    n_heads_kv: int,
    d_model: int,
    seq_len: int,
    batch_size: int = 1,
    n_warmup: int = 5,
    n_trials: int = 20,
) -> dict:
    """Benchmark one forward pass of GQA for a single decode step."""
    layer = GroupedQueryAttention(d_model, n_heads_q, n_heads_kv).to(device)
    layer.eval()

    # Simulate having a full KV cache (seq_len - 1 prior tokens)
    # and generating token number seq_len
    # We pass the full context at once for timing purposes
    x = torch.randn(batch_size, seq_len, d_model, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = layer(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Time
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            for _ in range(n_trials):
                _ = layer(x)
        end.record()
        torch.cuda.synchronize()
        elapsed_ms = start.elapsed_time(end) / n_trials
        peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6
    else:
        import time
        start = time.perf_counter()
        with torch.no_grad():
            for _ in range(n_trials):
                _ = layer(x)
        elapsed_ms = (time.perf_counter() - start) / n_trials * 1000
        peak_mem_mb = 0.0

    # KV cache theoretical size (for one layer, what we computed analytically)
    kv_cache_mb = (2 * n_heads_kv * seq_len * (d_model // n_heads_q) * 2) / 1e6

    return {
        'n_kv_heads':    n_heads_kv,
        'elapsed_ms':    elapsed_ms,
        'peak_mem_mb':   peak_mem_mb,
        'kv_cache_mb':   kv_cache_mb,
        'compression':   n_heads_kv / n_heads_q,
    }

# ── Run benchmark ──────────────────────────────────────────────────────────────
print("Running attention benchmarks (this takes ~1 minute)...")
D_MODEL   = 512
N_HEADS_Q = 16
SEQ_LEN   = 256

benchmark_configs = [
    ('MHA  (G=16)', N_HEADS_Q),
    ('GQA  (G=8)',   8),
    ('GQA  (G=4)',   4),
    ('GQA  (G=2)',   2),
    ('MQA  (G=1)',   1),
]

bench_results = []
for name, n_kv in benchmark_configs:
    res = measure_attention_memory_and_time(
        n_heads_q=N_HEADS_Q, n_heads_kv=n_kv,
        d_model=D_MODEL, seq_len=SEQ_LEN
    )
    res['name'] = name
    bench_results.append(res)
    print(f"  {name}: {res['elapsed_ms']:.3f} ms | "
          f"peak mem {res['peak_mem_mb']:.1f} MB | "
          f"KV cache ~{res['kv_cache_mb']:.2f} MB")

print("\n✅ Benchmarks complete!")

In [ ]:
# ── Visualize benchmark results ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

names    = [r['name'] for r in bench_results]
times    = [r['elapsed_ms'] for r in bench_results]
mems     = [r['peak_mem_mb'] for r in bench_results]
kv_mbs   = [r['kv_cache_mb'] for r in bench_results]
n_kvs    = [r['n_kv_heads'] for r in bench_results]

bar_colors = plt.cm.RdYlGn(np.linspace(0.2, 0.85, len(names)))

for ax, vals, ylabel, title in [
    (axes[0], times,  'Forward pass time (ms)',  'Latency vs KV Head Count'),
    (axes[1], mems,   'Peak Memory (MB)',         'Peak Memory Usage'),
    (axes[2], kv_mbs, 'Theoretical KV Cache (MB)', 'Analytical KV Cache Size'),
]:
    bars = ax.bar(names, vals, color=bar_colors, edgecolor='black', linewidth=1.2)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'GQA Benchmark: d_model={D_MODEL}, n_q={N_HEADS_Q}, T={SEQ_LEN}\n'
             '(measured on your Colab GPU)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('gqa_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Benchmark results saved as gqa_benchmark.png")

## 8. 🎯 Final Output — The Complete Bandwidth Fight Dashboard

Let us produce the definitive visualization that ties together everything we have learned: roofline position, memory savings, latency estimates, and quantization error — all in one screenshot-worthy panel.

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

# ── Panel A: Annotated Roofline ───────────────────────────────────────────────
ax_roof = fig.add_subplot(gs[0, :2])

ai_vals     = np.logspace(-1, 3, 500)
peak_compute = T4_PEAK_TFLOPS_FP16 * 1e12
peak_bw      = T4_BANDWIDTH_TBPS   * 1e12
perf         = np.minimum(ai_vals * peak_bw, peak_compute)

ax_roof.loglog(ai_vals, perf / 1e12, color='steelblue', linewidth=3)
ax_roof.axvspan(ai_vals[0], T4_RIDGE_POINT, alpha=0.07, color='red')
ax_roof.axvspan(T4_RIDGE_POINT, ai_vals[-1], alpha=0.07, color='green')
ax_roof.axvline(T4_RIDGE_POINT, color='gray', linestyle='--', alpha=0.5)

roofline_technique_points = [
    ('MHA fp16',       1.2,  'crimson'),
    ('GQA fp16',       4.8,  'darkorange'),
    ('MQA fp16',       9.6,  'gold'),
    ('GQA int8',       9.6,  'limegreen'),
    ('GQA int4',       19.2, 'dodgerblue'),
]
for label, ai, color in roofline_technique_points:
    p_pt = min(ai * peak_bw, peak_compute) / 1e12
    ax_roof.scatter(ai, p_pt, s=180, color=color, zorder=5,
                    edgecolors='black', linewidth=1.5)
    ax_roof.annotate(label, (ai, p_pt), textcoords='offset points',
                     xytext=(6, 5), fontsize=9, fontweight='bold', color=color)

# Arrow showing the "bandwidth fight" direction
ax_roof.annotate('', xy=(15, 2.5), xytext=(1.5, 0.3),
                 arrowprops=dict(arrowstyle='->', color='purple',
                                 lw=2.5, connectionstyle='arc3,rad=-0.3'))
ax_roof.text(4.5, 0.7, 'GQA + Quantization\npushes us right →',
             fontsize=9, color='purple', fontstyle='italic')

ax_roof.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=11)
ax_roof.set_ylabel('Performance (TFLOPS)', fontsize=11)
ax_roof.set_title('The Bandwidth Fight — Roofline View (T4 GPU)', fontsize=11, fontweight='bold')
ax_roof.grid(True, which='both', alpha=0.25)

# ── Panel B: Technique comparison radar / table ───────────────────────────────
ax_table = fig.add_subplot(gs[0, 2])
ax_table.axis('off')

T_demo = 4096
table_data = []
mha_lat_ref = estimate_token_latency_ms(llama2_7b_mha, T_demo)
for name, cfg in configs.items():
    lat = estimate_token_latency_ms(cfg, T_demo)
    gb  = cfg.kv_cache_gb(T_demo)
    spd = mha_lat_ref / lat
    table_data.append([name, f'{gb:.2f} GB', f'{lat:.2f} ms', f'{spd:.1f}×'])

col_labels = ['Technique', 'Cache (4K)', 'Latency', 'Speedup']
table = ax_table.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.7)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#f0f4f8')
    cell.set_edgecolor('white')

ax_table.set_title('Summary at T=4K tokens\n(LLaMA-2 7B arch.)',
                   fontsize=10, fontweight='bold', pad=12)

# ── Panel C: Cache size growth ────────────────────────────────────────────────
ax_cache = fig.add_subplot(gs[1, :2])
T_rng = np.arange(512, 17000, 256)
for name, cfg in configs.items():
    gb_vals = [cfg.kv_cache_gb(int(T)) for T in T_rng]
    ax_cache.plot(T_rng / 1000, gb_vals, color=colors_map[name],
                  linewidth=2.5, label=name)
ax_cache.axhline(16, color='black', linestyle=':', alpha=0.5)
ax_cache.text(1, 16.3, 'T4 VRAM ≈ 16 GB', fontsize=8, color='gray')
ax_cache.set_xlabel('Sequence Length (K tokens)', fontsize=10)
ax_cache.set_ylabel('KV Cache Size (GB)', fontsize=10)
ax_cache.set_title('KV Cache Memory — Every Technique Reduces This', fontsize=10, fontweight='bold')
ax_cache.legend(fontsize=8, loc='upper left')
ax_cache.grid(alpha=0.3)

# ── Panel D: INT8 quantization error ─────────────────────────────────────────
ax_q8 = fig.add_subplot(gs[1, 2])
K_demo = torch.randn(1, 4, 128, 16) * 0.5
cache8 = QuantizedKVCache(bits=8)
cache8.update(K_demo, K_demo)
K_rec8, _ = cache8.read()
err8 = (K_rec8 - K_demo).flatten().numpy()
ax_q8.hist(err8, bins=60, color='limegreen', edgecolor='white', alpha=0.85)
ax_q8.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax_q8.set_xlabel('Quantization Error', fontsize=10)
ax_q8.set_ylabel('Count', fontsize=10)
ax_q8.set_title(f'INT8 Quant Error\nσ = {err8.std():.5f}', fontsize=10, fontweight='bold')
ax_q8.grid(alpha=0.3)

# ── Panel E: Throughput curves ────────────────────────────────────────────────
ax_tps2 = fig.add_subplot(gs[2, :])
T_rng2 = np.arange(512, 33000, 512)
for name, cfg in configs.items():
    tps_vals = [estimate_max_throughput_tps(cfg, int(T)) for T in T_rng2]
    ax_tps2.plot(T_rng2 / 1000, tps_vals, color=colors_map[name],
                 linewidth=2.5, label=name)

ax_tps2.set_xlabel('Sequence Length (K tokens)', fontsize=11)
ax_tps2.set_ylabel('Bandwidth-Limited Throughput (tokens/sec)', fontsize=11)
ax_tps2.set_title('Peak Throughput vs Sequence Length — The Bandwidth Fight in Numbers',
                  fontsize=11, fontweight='bold')
ax_tps2.legend(fontsize=9, loc='upper right')
ax_tps2.grid(alpha=0.3)

# Shade the "usable" region
ax_tps2.fill_between(T_rng2 / 1000,
                     [estimate_max_throughput_tps(llama2_7b_gqa_int4, int(T)) for T in T_rng2],
                     [estimate_max_throughput_tps(llama2_7b_mha, int(T))      for T in T_rng2],
                     alpha=0.08, color='dodgerblue',
                     label='Gain from GQA+INT4 vs MHA')

plt.suptitle('Section 5: Living With the Necessary Evil\n'
             'GQA + Quantization — The Complete Bandwidth Fight Dashboard',
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('bandwidth_fight_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("🎯 Final dashboard saved as bandwidth_fight_dashboard.png")
print("   This is your screenshot-worthy summary of the entire section!")

## 9. Reflection and Next Steps

### What We Built

Let us take stock of what we have accomplished in this notebook:

1. **The Roofline Intuition** — We built a working roofline visualizer and saw quantitatively that LLM inference lives deep in the memory-bound regime. Every technique in the KV cache design space is trying to move that operating point rightward.

2. **The Memory Calculator** — We derived and implemented the exact formula for KV cache memory as a function of heads, layers, sequence length, and precision. We saw that a 7B model's MHA cache at 32K tokens would overflow a T4 entirely.

3. **Grouped-Query Attention** — We implemented GQA from scratch, showing how query heads share KV heads, reducing cache by the ratio $G/H$. The `_repeat_kv` trick is the key: KV heads are expanded at compute time, not stored time.

4. **Quantization** — We implemented absmax INT8/INT4 quantization and measured the reconstruction error. INT8 is near-lossless for typical activation distributions; INT4 is coarser but still remarkably usable.

5. **The Combined Dashboard** — We showed that GQA and quantization are orthogonal: their savings multiply. GQA G=8 + INT4 gives a 32× cache reduction versus MHA FP16.

### 🤔 Reflection Questions

**Question 1:** We showed that GQA G=8 and INT8 quantization both give roughly 4× reduction in bandwidth demand (for a 32-head model). But they achieve this in fundamentally different ways. Can you articulate the *qualitative* difference in what each technique sacrifices?

**Question 2:** Our quantization implementation uses *per-tensor* absmax scaling. Production systems use *per-channel* or *per-group* scaling. Why would finer-grained scaling reduce quantization error? What is the tradeoff?

**Question 3:** We have been assuming the bottleneck is the KV *read* bandwidth. But there is also the model *weight* bandwidth — reading the transformer's parameter matrices. At what sequence length does KV bandwidth overtake weight bandwidth as the dominant cost? (Hint: think about how many bytes of weights versus KV cache the model reads per token.)

**Question 4:** GQA reduces the *number* of KV heads but not the *head dimension* $d_h$. An alternative design would keep all heads but reduce $d_h$. Would these two strategies have the same effect on the cache? Would they have the same effect on model quality?

### 🏆 Optional Challenges

**Challenge 1 — FlashAttention-style tiling:**
Our GQA implementation loads the full K and V matrices into memory before computing attention. Real systems use *tiled* computation to fuse the KV read and attention arithmetic into a single kernel, avoiding a round-trip through VRAM. Can you sketch (in pseudocode or code) how you would tile the attention computation to reduce peak memory?

**Challenge 2 — Per-channel quantization:**
Extend `QuantizedKVCache` to use per-head quantization: each of the `n_kv` heads gets its own scale factor. Measure the reduction in mean absolute error compared to the per-tensor version.

**Challenge 3 — GQA with rotary embeddings:**
Real models (LLaMA, Mistral) apply Rotary Position Embeddings (RoPE) to the Q and K projections before attention. Extend your `GroupedQueryAttention` to apply RoPE. The key subtlety: the position encoding must match the *actual* token position, not the position within the current input chunk.

**Challenge 4 — Simulate a real serving scenario:**
Build a simulation that generates 200 tokens autoregressively using the `kv_cache` argument in `GroupedQueryAttention.forward()`. Plot the per-token latency as the cache grows. Does it match our theoretical $t \propto T$ prediction?

### 📚 Where to Go From Here

The techniques we studied in this notebook are deployed in every major LLM serving system today:

- **GQA** is used in LLaMA-2 70B, Mistral 7B, Gemma, and most modern architectures
- **INT8/INT4 KV quantization** is a standard option in vLLM, TGI, and TensorRT-LLM
- **Paged attention** (not covered here) addresses *fragmentation* — ensuring the truck's cargo hold is always full, not partially wasted on reserved-but-unused slots
- **Speculative decoding** attacks the problem from a completely different angle: generate multiple tokens per KV read by using a cheap draft model

Each of these sits on the same roofline, playing the same game: more information per byte moved across that precious memory bus.

The bandwidth fight is ongoing. And now you are equipped to follow it.

---

*🎓 This notebook is part of the Vizuara series "The Good and Evil of the Key-Value Cache".*
*Continue to Section 6: Paged Attention and the Memory Fragmentation Problem.*